<a href="https://colab.research.google.com/github/sreelekha2196/Pixel-Seal-Hybrid-Watermarking-Framework-/blob/main/notebooks/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Copyright (c) Meta Platforms, Inc. and affiliates. All rights reserved. This source code is licensed under the license found in the LICENSE file in the root directory of this source tree.

# Video Seal Inference

[[`arXiv`](https://arxiv.org/abs/2412.09492)]
[[`Colab`](https://colab.research.google.com/github/facebookresearch/videoseal/blob/main/notebooks/colab.ipynb)]
[[`Demo`](https://aidemos.meta.com/videoseal)]

## Installation

Clone repository and install dependencies

In [1]:
!git clone https://github.com/facebookresearch/videoseal.git
%cd videoseal

Cloning into 'videoseal'...
remote: Enumerating objects: 682, done.
remote: Counting objects: 100% (323/323), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 682 (delta 182), reused 160 (delta 157), pack-reused 359 (from 2)
Receiving objects: 100% (682/682), 27.83 MiB | 18.12 MiB/s, done.
Resolving deltas: 100% (306/306), done.
/content/videoseal


Install dependencies

In [2]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.2/226.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 110.7 MB/s eta 0:00:00
  Attempting uninstall: timm
    Found existing installation: timm 1.0.29
    Uninstalling timm-1.0.29:
      Successfully uninstalled timm-1.0.29


## Imports and loading

In [3]:
%cd /content/videoseal

/content/videoseal


In [4]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import logging
logging.getLogger("matplotlib.image").setLevel(logging.ERROR)
from IPython.display import HTML, display

import pandas as pd
from tqdm import tqdm
import numpy as np
import ffmpeg
import os
import cv2
import subprocess

import torch

from videoseal.evals.metrics import bit_accuracy
from videoseal.models import Videoseal
from videoseal.utils.cfg import setup_model_from_model_card


def get_video_info(input_path):
    # Open the video file
    video = cv2.VideoCapture(input_path)

    # Get video properties
    width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = video.get(cv2.CAP_PROP_FPS)
    codec = int(video.get(cv2.CAP_PROP_FOURCC))
    num_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))

    # Decode codec to human-readable form
    codec_str = "".join([chr((codec >> 8 * i) & 0xFF) for i in range(4)])

    video.release()  # Close the video file

    return {
        "width": width,
        "height": height,
        "fps": fps,
        "codec": codec_str,
        "num_frames": num_frames
    }

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/content/videoseal/videoseal/models/baselines.py:339: SyntaxWarning: invalid escape sequence '\;'
  `pip install huggingface_hub; huggingface-cli download tangtianzhong/img-wm-torchscript --cache-dir .cache; mkdir ckpts; find .cache/models--tangtianzhong--img-wm-torchscript/snapshots/845dc751783db2a03a4b14ea600b0a4a9aba89aa -type l -exec cp --dereference {} ckpts/ \; sleep 5 ;rm -rf .cache`


## Load the model

The videoseal library provides pretrained models for embedding and extracting watermarks.

In [5]:
# Load the VideoSeal model
model = setup_model_from_model_card("videoseal")

# Set the model to evaluation mode and move it to the selected device
model = model.eval()
model = model.to(device)
model.compile()

# Setup the step size. Bigger step size makes embedding faster but loses a bit of robustness.
model.step_size = 8

File https://dl.fbaipublicfiles.com/videoseal/y_256b_img.pth downloaded successfully to /content/videoseal/ckpts/videoseal_y_256b_img.pth
Model loaded successfully from /content/videoseal/ckpts/videoseal_y_256b_img.pth with message: <All keys matched successfully>


## Embedding

The embedding process is the process of hiding the watermark in the video.

In [6]:
def embed_video_clip(
    model: Videoseal,
    clip: np.ndarray,
    msgs: torch.Tensor
) -> np.ndarray:
    clip_tensor = torch.tensor(clip, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    outputs = model.embed(clip_tensor, msgs=msgs, is_video=True, lowres_attenuation=True)
    processed_clip = outputs["imgs_w"]
    processed_clip = (processed_clip * 255.0).byte().permute(0, 2, 3, 1).numpy()
    return processed_clip

def embed_video(
    model: Videoseal,
    input_path: str,
    output_path: str,
    chunk_size: int,
    crf: int = 23
) -> None:
    # Read video dimensions
    video_info = get_video_info(input_path)
    width = int(video_info['width'])
    height = int(video_info['height'])
    fps = float(video_info['fps'])
    codec = video_info['codec']
    num_frames = int(video_info['num_frames'])

    # Open the input video
    process1 = (
        ffmpeg
        .input(input_path)
        .output('pipe:', format='rawvideo', pix_fmt='rgb24', s='{}x{}'.format(width, height), r=fps)
        .run_async(pipe_stdout=True, pipe_stderr=False)
    )
    # Open the output video
    process2 = (
        ffmpeg
        .input('pipe:', format='rawvideo', pix_fmt='rgb24', s='{}x{}'.format(width, height), r=fps)
        .output(output_path, vcodec='libx264', pix_fmt='yuv420p', r=fps, crf=crf)
        .overwrite_output()
        .run_async(pipe_stdin=True, pipe_stderr=False)
    )

    # Create a random message
    msgs = model.get_random_msg()
    with open(output_path.replace(".mp4", ".txt"), "w") as f:
        f.write("".join([str(msg.item()) for msg in msgs[0]]))

    # Process the video
    frame_size = width * height * 3
    chunk = np.zeros((chunk_size, height, width, 3), dtype=np.uint8)
    frame_count = 0
    pbar = tqdm(total=num_frames, desc="Watermark embedding")
    while True:
        in_bytes = process1.stdout.read(frame_size)
        if not in_bytes:
            break
        frame = np.frombuffer(in_bytes, np.uint8).reshape([height, width, 3])
        chunk[frame_count % chunk_size] = frame
        frame_count += 1
        pbar.update(1)
        if frame_count % chunk_size == 0:
            processed_frame = embed_video_clip(model, chunk, msgs)
            process2.stdin.write(processed_frame.tobytes())
    process1.stdout.close()
    process2.stdin.close()
    process1.wait()
    process2.wait()

    return msgs

You are free to upload any video and change the `video_path`.

You can look at the watermark video output in the folder `outputs`.

In [7]:
# Path to the input video
video_path = "./assets/videos/1.mp4"

# Create the output directory and path
output_dir = "./outputs"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, os.path.basename(video_path))

# Embed the watermark inside the video with a random msg
msgs_ori = embed_video(model, video_path, output_path, 16)
print(f"\nSaved watermarked video to {output_path}")

Watermark embedding: 100%|██████████| 256/256 [02:19<00:00,  1.84it/s]


Saved watermarked video to ./outputs/1.mp4


## Extraction

Load the video output from the embedding process and extract the watermark.

In [8]:
def detect_video_clip(
    model: Videoseal,
    clip: np.ndarray
) -> torch.Tensor:
    clip_tensor = torch.tensor(clip, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
    outputs = model.detect(clip_tensor, is_video=True)
    output_bits = outputs["preds"][:, 1:]  # exclude the first which may be used for detection
    return output_bits

def detect_video(
    model: Videoseal,
    input_path: str,
    num_frames_for_extraction: int,
    chunk_size: int
) -> None:
    # Read video dimensions
    video_info = get_video_info(input_path)
    width = int(video_info['width'])
    height = int(video_info['height'])
    num_frames = int(video_info['num_frames'])

    # Open the input video
    process1 = (
        ffmpeg
        .input(input_path)
        .output('pipe:', format='rawvideo', pix_fmt='rgb24')
        .run_async(pipe_stdout=True, pipe_stderr=False)
    )

    # Process the video
    frame_size = width * height * 3
    chunk = np.zeros((chunk_size, height, width, 3), dtype=np.uint8)
    frame_count = 0
    soft_msgs = []
    pbar = tqdm(total=num_frames, desc="Watermark extraction")
    while True:
        if frame_count >= num_frames_for_extraction:
            break
        in_bytes = process1.stdout.read(frame_size)
        if not in_bytes:
            break
        frame = np.frombuffer(in_bytes, np.uint8).reshape([height, width, 3])
        chunk[frame_count % chunk_size] = frame
        frame_count += 1
        pbar.update(1)
        if frame_count % chunk_size == 0:
            soft_msgs.append(detect_video_clip(model, chunk))
    process1.stdout.close()
    process1.wait()

    soft_msgs = torch.cat(soft_msgs, dim=0)
    soft_msgs = soft_msgs.mean(dim=0)  # Average the predictions across all frames
    return soft_msgs

In [9]:
# Detect the watermark
num_frames_for_extraction = 32
soft_msgs = detect_video(model, output_path, num_frames_for_extraction, 16)
bit_acc = bit_accuracy(soft_msgs, msgs_ori).item() * 100
print(f"\nBinary message extracted with {bit_acc:.1f}% bit accuracy")

Watermark extraction:  12%|█▎        | 32/256 [00:17<02:03,  1.81it/s]


Binary message extracted with 98.8% bit accuracy


## Run other baselines

To download other checkpoints, you can run the following command:

In [10]:
!pip install huggingface_hub
!huggingface-cli download tangtianzhong/img-wm-torchscript --cache-dir .cache
!mkdir ckpts
!cp .cache/models--tangtianzhong--img-wm-torchscript/snapshots/845dc751783db2a03a4b14ea600b0a4a9aba89aa/*.pt ckpts/
!rm -rf .cache


Hint: A new version of huggingface_hub (1.32.0) is available! You are using version 1.29.0.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help

mkdir: cannot create directory ‘ckpts’: File exists
cp: cannot stat '.cache/models--tangtianzhong--img-wm-torchscript/snapshots/845dc751783db2a03a4b14ea600b0a4a9aba89aa/*.pt': No such file or directory


In [11]:
from videoseal.utils.cfg import setup_model_from_checkpoint

model = setup_model_from_checkpoint("baseline/trustmark")
model = model.eval()
model = model.to(device)
model.compile()

model.chunk_size = 32  # embed 32 frames/imgs at a time
model.step_size = 4  # propagate the wm to 4 next frame/img
# model.blender.scaling_w *= 1.5  # imperceptibility/robustness trade-off

AssertionError: 
Please download the baseline models first.  
See docs/baselines.md for instructions, or run:
`pip install huggingface_hub; huggingface-cli download tangtianzhong/img-wm-torchscript --cache-dir .cache; mkdir ckpts; find .cache/models--tangtianzhong--img-wm-torchscript/snapshots/845dc751783db2a03a4b14ea600b0a4a9aba89aa -type l -exec cp --dereference {} ckpts/ \; sleep 5 ;rm -rf .cache`


### Embedding

In [ ]:
# Path to the input video
video_path = "./assets/videos/1.mp4"

# Create the output directory and path
output_dir = "./outputs"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, os.path.basename(video_path))

# Embed the watermark inside the video with a random msg
msgs_ori = embed_video(model, video_path, output_path, 16)
print(f"\nSaved watermarked video to {output_path}")

### Extraction

In [12]:
import videoseal
from PIL import Image
import torchvision.transforms as T

model = videoseal.load("pixelseal")
model = model.eval()



File https://dl.fbaipublicfiles.com/videoseal/pixelseal/checkpoint.pth downloaded successfully to /content/videoseal/ckpts/pixelseal_checkpoint.pth
Model loaded successfully from /content/videoseal/ckpts/pixelseal_checkpoint.pth with message: <All keys matched successfully>
